# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to programmatically explore, extract, and analyze the [FAIR² Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant-python/) library.

You will learn to load Croissant metadata, review available record sets, access and manipulate data, and perform basic exploratory analysis leveraging record, field, and column `@id` references per the FAIR standard.

### Dataset Source
*Croissant schema URL*: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
# (uncomment the following line if running in a new environment)
!pip install -U mlcroissant

## 1. Data Loading

Load dataset metadata and structure using the Croissant schema and `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset overview from metadata (using attributes, not dict subscripting)
meta = dataset.metadata
print(f"\033[1m{meta.name}\033[0m\n{meta.description}")
print(f"DOI: {getattr(meta, 'identifier', None)}")
print(f"License: {getattr(meta, 'license', None)}")
print(f"Published: {getattr(meta, 'datePublished', None)}")

## 2. Data Overview

Examine the available record sets and discover available field `@id`s for referencing specific data.

In [ ]:
# List all record sets (table-like structures) in the dataset
print("\033[1mAvailable Record Sets (@id and name):\033[0m")
record_sets = []
if hasattr(meta, 'recordSet') and meta.recordSet:
    # meta.recordSet could be dict or list
    rs_list = meta.recordSet if isinstance(meta.recordSet, list) else [meta.recordSet]
    for rs in rs_list:
        # The mlcroissant API provides .@id and .name for record sets
        print(f"  - @id: {getattr(rs, '@id', None)}, name: {getattr(rs, 'name', None)}")
        record_sets.append(getattr(rs, '@id', None))
else:
    print("No record sets found directly in metadata. Trying auto-discovery via dataset API...")
    # Attempt to discover record set IDs via dataset._record_set_ids (private API fallback, non-ideal)
    rs_ids = getattr(dataset, '_record_set_ids', None)
    if rs_ids:
        for rsid in rs_ids:
            print(f"  - @id: {rsid}")
            record_sets.append(rsid)
    else:
        print("No record sets found.")

# For each record set, list available field and column @ids
print("\n\033[1mFields per Record Set (@id, name, dataType):\033[0m")
for rsid in record_sets:
    print(f"\nRecordSet: {rsid}")
    # The mlcroissant API lets you inspect record set structure:
    try:
        rs_struct = dataset._find_by_id(rsid)
    except Exception:
        rs_struct = None
    if rs_struct and hasattr(rs_struct, 'field') and rs_struct.field:
        fields = rs_struct.field if isinstance(rs_struct.field, list) else [rs_struct.field]
        for f in fields:
            print(f"  - Field @id: {getattr(f, '@id', None)}\tname: {getattr(f, 'name', None)}\ttype: {getattr(f, 'dataType', None)}")
    else:
        print("  No fields found for this record set.")

## 3. Data Extraction

Load records from each available record set using the `@id` of the record set. Each record uses keys named by its field `@id`. Data is loaded into a pandas `DataFrame` for analysis.

In [ ]:
dataframes = {}
if not record_sets:
    print("No record sets available for extraction.")
else:
    for record_set_id in record_sets:
        print(f"\nExtracting records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns (@id): {list(df.columns)}")
            display(df.head())
        else:
            print("  No records loaded for this record set.")

## 4. Exploratory Data Analysis (EDA)

We'll identify a numeric field (using its `@id`) and a grouping field, then demonstrate filtering, normalization, and aggregation. Adjust `numeric_field_id` and `group_field_id` as discovered in your dataset above.

In [ ]:
# Example: Automated selection of a numeric field and a group field
import numpy as np

if not dataframes:
    print("No dataframes available for EDA.")
else:
    # Take the first record set with data
    rsid = next(iter(dataframes.keys()))
    df = dataframes[rsid]
    # Try to heuristically pick a numeric field (@id)
    # We'll look for columns with integer or float types and not all unique
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > 3:
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Try to guess by name
        for col in df.columns:
            if 'age' in col.lower() or 'interval' in col.lower() or 'count' in col.lower():
                try:
                    # Try to coerce to numeric
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    numeric_field_id = col
                    break
                except Exception:
                    pass
    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}")
        # Filter example: threshold at 60th percentile
        threshold = df[numeric_field_id].quantile(0.6) if df[numeric_field_id].dtype != object else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df[[numeric_field_id]].head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Attempt to find a group field (@id): pick a categorical with low cardinality
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and 2 < df[col].nunique() < 15:
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
            display(grouped_df)
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field detected for EDA. Inspect columns and customize field selection as needed.")

## 5. Visualization

Visualize distributions and relationships. Example: plot histogram for the selected numeric field and a boxplot by a categorical group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color="skyblue")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, palette="Set2")
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to load a FAIR² dataset described by a Croissant schema using the `mlcroissant` library and reference all record sets and fields by their `@id` as required by the standard.
- We provided an overview of available data structures, loaded records, performed initial processing (filtering, normalization, grouping), and visualized selected distributions.
- To extend the analysis: review available field `@id`s, select relevant variables for your research, and build custom processing or modeling pipelines, always using persistent `@id` references for robust code.